# BTC Regime Gate v2.1 — Google Colab Backtest

1. Upload `BTCUSDT_2025_full.xlsx`
2. Run all cells
3. Starting capital = **100,000 USDT** — auto buy/sell using the same playbooks as TradingView

Data is 5m OHLCV → resampled to **15m** (matches Pine), HTF = **1h**.

In [ ]:
!pip -q install pandas openpyxl numpy matplotlib

from google.colab import files
uploaded = files.upload()  # choose BTCUSDT_2025_full.xlsx
DATA_PATH = list(uploaded.keys())[0]
print('Using', DATA_PATH)

In [ ]:
# If you already uploaded, set path manually:
# DATA_PATH = '/content/BTCUSDT_2025_full.xlsx'

import math
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STARTING_CAPITAL = 100_000.0
RISK_PER_TRADE = 0.01
MAX_POSITION_FRAC = 0.20
SLIPPAGE_BPS = 9.0
FEE_BPS_ONE_WAY = 4.0

ADX_TREND, ADX_RANGE = 25.0, 20.0
VOL_SHOCK_MULT = 2.5
PULLBACK_TOL = 0.50
ATR_SL_TREND, ATR_TRAIL_TREND, BE_AT_R = 1.5, 2.0, 1.0
MAX_HOLD_TREND = 48
VWAP_Z_ENTRY, VWAP_Z_EXIT = 1.8, 0.3
RSI_OS, RSI_OB = 30.0, 70.0
ATR_SL_RANGE, ATR_TP_RANGE = 1.0, 1.5
MAX_HOLD_RANGE = 16
RVOL_MIN, SCORE_LONG = 0.8, 60.0
COOLDOWN_BARS, MIN_HOLD_BARS = 8, 3

def ema(s, n): return s.ewm(span=n, adjust=False).mean()
def sma(s, n): return s.rolling(n).mean()

def rsi(close, n=14):
    d = close.diff(); up = d.clip(lower=0); dn = -d.clip(upper=0)
    au = up.ewm(alpha=1/n, adjust=False).mean(); ad = dn.ewm(alpha=1/n, adjust=False).mean()
    rs = au / ad.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def atr(df, n=14):
    prev = df['close'].shift(1)
    tr = pd.concat([(df['high']-df['low']).abs(), (df['high']-prev).abs(), (df['low']-prev).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/n, adjust=False).mean()

def adx(df, n=14):
    up = df['high'].diff(); dn = -df['low'].diff()
    plus_dm = np.where((up > dn) & (up > 0), up, 0.0)
    minus_dm = np.where((dn > up) & (dn > 0), dn, 0.0)
    prev_c = df['close'].shift(1)
    tr_raw = pd.concat([(df['high']-df['low']).abs(), (df['high']-prev_c).abs(), (df['low']-prev_c).abs()], axis=1).max(axis=1)
    atr_w = tr_raw.ewm(alpha=1/n, adjust=False).mean()
    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/n, adjust=False).mean() / atr_w.replace(0, np.nan)
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/n, adjust=False).mean() / atr_w.replace(0, np.nan)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    return dx.ewm(alpha=1/n, adjust=False).mean()

raw = pd.read_excel(DATA_PATH)
raw = raw.rename(columns={'Open Time':'timestamp','Open':'open','High':'high','Low':'low','Close':'close','Volume':'volume'})
raw['timestamp'] = pd.to_datetime(raw['timestamp'])
raw = raw.sort_values('timestamp').drop_duplicates('timestamp').set_index('timestamp')
raw = raw[['open','high','low','close','volume']].astype(float)
print('5m bars:', len(raw))

def resample_ohlcv(df, rule):
    return df.resample(rule, label='left', closed='left').agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'}).dropna()

df15 = resample_ohlcv(raw, '15min')
df1h = resample_ohlcv(raw, '1h')
print('15m bars:', len(df15), '| 1h bars:', len(df1h))

In [ ]:
d = df15.copy()
d['atr'] = atr(d, 14); d['atr_ma'] = sma(d['atr'], 50); d['adx'] = adx(d, 14)
d['ema21'] = ema(d['close'], 21); d['ema50'] = ema(d['close'], 50)
d['ema_slope'] = ema(d['close'], 10); d['kf_vel'] = d['ema_slope'] - d['ema_slope'].shift(1)
d['vol_sma'] = sma(d['volume'], 20); d['rvol'] = np.where(d['vol_sma']>0, d['volume']/d['vol_sma'], 0.0)
d['rsi'] = rsi(d['close'], 14)
typ = (d['high']+d['low']+d['close'])/3
vol_sum = d['volume'].rolling(48).sum(); typ_vol = (typ*d['volume']).rolling(48).sum()
d['roll_vwap'] = np.where(vol_sum>0, typ_vol/vol_sum, d['close'])
d['vwap_z'] = (d['close']-d['roll_vwap']) / (d['close']-d['roll_vwap']).rolling(48).std().replace(0, np.nan)
htf = df1h.copy(); htf['ema50']=ema(htf['close'],50); htf['ema200']=ema(htf['close'],200)
htf_flag = (htf['ema50']>htf['ema200']).astype(float).shift(1)
d['htf_up'] = htf_flag.reindex(d.index, method='ffill')
d['htf_dn'] = (1.0 - d['htf_up']).where(d['htf_up'].notna(), np.nan)
d['vol_shock'] = (d['atr_ma']>0) & ((d['atr']/d['atr_ma'])>=VOL_SHOCK_MULT)
d['is_trend'] = (~d['vol_shock']) & (d['adx']>=ADX_TREND)
d['is_range'] = (~d['vol_shock']) & (d['adx']<=ADX_RANGE)
near_long = (d['low']<=d['ema21']+PULLBACK_TOL*d['atr']) | ((d['close']-d['ema21']).abs()<=PULLBACK_TOL*d['atr'])
reclaim_long = (d['close']>d['ema21']) & (d['close']>=d['open']); struct_up = d['ema21']>d['ema50']
d['long_hard_a'] = d['is_trend']&(d['htf_up']==1)&struct_up&near_long&reclaim_long&(d['rvol']>=RVOL_MIN)&(d['kf_vel']>=0)
near_short = (d['high']>=d['ema21']-PULLBACK_TOL*d['atr']) | ((d['close']-d['ema21']).abs()<=PULLBACK_TOL*d['atr'])
reclaim_short = (d['close']<d['ema21']) & (d['close']<=d['open']); struct_dn = d['ema21']<d['ema50']
d['short_hard_a'] = d['is_trend']&(d['htf_dn']==1)&struct_dn&near_short&reclaim_short&(d['rvol']>=RVOL_MIN)&(d['kf_vel']<=0)
score_a_long = d['long_hard_a'].astype(float)*40+(d['htf_up']==1).astype(float)*20+(d['rvol']>=RVOL_MIN).astype(float)*15+(d['kf_vel']>=0).astype(float)*10+struct_up.astype(float)*10
score_a_short = d['short_hard_a'].astype(float)*40+(d['htf_dn']==1).astype(float)*20+(d['rvol']>=RVOL_MIN).astype(float)*15+(d['kf_vel']<=0).astype(float)*10+struct_dn.astype(float)*10
d['long_sig_a'] = d['long_hard_a'] & (score_a_long>=SCORE_LONG)
d['short_sig_a'] = d['short_hard_a'] & (score_a_short>=SCORE_LONG)
rev_up = (d['close']>d['open'])|(d['close']>d['close'].shift(1)); rev_dn=(d['close']<d['open'])|(d['close']<d['close'].shift(1))
d['long_hard_b'] = d['is_range']&(d['vwap_z']<=-VWAP_Z_ENTRY)&(d['rsi']<=RSI_OS)&rev_up&(d['rvol']>=RVOL_MIN)
d['short_hard_b'] = d['is_range']&(d['vwap_z']>=VWAP_Z_ENTRY)&(d['rsi']>=RSI_OB)&rev_dn&(d['rvol']>=RVOL_MIN)
score_b_long = d['long_hard_b'].astype(float)*40+(d['vwap_z']<=-VWAP_Z_ENTRY).astype(float)*20+(d['rvol']>=RVOL_MIN).astype(float)*15+(d['rsi']<=RSI_OS).astype(float)*15+rev_up.astype(float)*10
score_b_short = d['short_hard_b'].astype(float)*40+(d['vwap_z']>=VWAP_Z_ENTRY).astype(float)*20+(d['rvol']>=RVOL_MIN).astype(float)*15+(d['rsi']>=RSI_OB).astype(float)*15+rev_dn.astype(float)*10
d['long_sig_b'] = d['long_hard_b'] & (score_b_long>=SCORE_LONG)
d['short_sig_b'] = d['short_hard_b'] & (score_b_short>=SCORE_LONG)
feats = d.iloc[250:].copy()
print('Ready bars:', len(feats))
print('Signals A long/short:', int(feats['long_sig_a'].sum()), int(feats['short_sig_a'].sum()))
print('Signals B long/short:', int(feats['long_sig_b'].sum()), int(feats['short_sig_b'].sum()))

In [ ]:
@dataclass
class Trade:
    side: str; playbook: str; entry_time: pd.Timestamp; entry_price: float
    exit_time: Optional[pd.Timestamp]=None; exit_price: Optional[float]=None
    qty: float=0.0; pnl: float=0.0; reason: str=''

@dataclass
class Engine:
    equity: float=STARTING_CAPITAL; cash: float=STARTING_CAPITAL; pos: int=0
    entry_bar: int=-1; entry_price: float=0.0; stop: float=0.0; tp: float=math.nan; trail: float=math.nan
    qty: float=0.0; last_exit_bar: int=-10000; playbook: str=''
    trades: list=field(default_factory=list); equity_curve: list=field(default_factory=list)
    def fee(self, notional): return notional * (FEE_BPS_ONE_WAY/10000)
    def slip_price(self, price, side, is_entry):
        half = (SLIPPAGE_BPS/2)/10000
        if side=='long': return price*(1+half) if is_entry else price*(1-half)
        return price*(1-half) if is_entry else price*(1+half)
    def size_from_stop(self, entry, stop):
        risk_cash = self.equity * RISK_PER_TRADE; stop_dist = abs(entry-stop)
        if stop_dist<=0: return 0.0
        qty = risk_cash/stop_dist; max_qty = (self.equity*MAX_POSITION_FRAC)/entry
        return max(0.0, min(qty, max_qty))
    def mark_equity(self, price):
        if self.pos in (1,3): return self.cash + self.qty*price
        if self.pos in (2,4): return self.cash - self.qty*price
        return self.cash

def run_backtest(d):
    eng = Engine(); idx=d.index; highs=d['high'].values; lows=d['low'].values; closes=d['close'].values
    atrs=d['atr'].values; vwap_z=d['vwap_z'].values
    long_a=d['long_sig_a'].fillna(False).values; short_a=d['short_sig_a'].fillna(False).values
    long_b=d['long_sig_b'].fillna(False).values; short_b=d['short_sig_b'].fillna(False).values
    for i in range(len(d)):
        if np.isnan(atrs[i]) or atrs[i]<=0:
            eng.equity_curve.append(eng.mark_equity(closes[i])); continue
        if eng.pos!=0:
            held=i-eng.entry_bar; can_exit=held>=MIN_HOLD_BARS; exit_px=None; reason=''
            if eng.pos==1 and can_exit:
                if highs[i]>=eng.entry_price+BE_AT_R*atrs[i]: eng.stop=max(eng.stop, eng.entry_price)
                eng.trail=max(eng.trail if not math.isnan(eng.trail) else closes[i]-ATR_TRAIL_TREND*atrs[i], closes[i]-ATR_TRAIL_TREND*atrs[i])
                eng.stop=max(eng.stop, eng.trail)
                if lows[i]<=eng.stop: exit_px, reason = eng.stop, 'A long stop/trail'
                elif held>=MAX_HOLD_TREND: exit_px, reason = closes[i], 'A long time'
            elif eng.pos==2 and can_exit:
                if lows[i]<=eng.entry_price-BE_AT_R*atrs[i]: eng.stop=min(eng.stop, eng.entry_price)
                eng.trail=min(eng.trail if not math.isnan(eng.trail) else closes[i]+ATR_TRAIL_TREND*atrs[i], closes[i]+ATR_TRAIL_TREND*atrs[i])
                eng.stop=min(eng.stop, eng.trail)
                if highs[i]>=eng.stop: exit_px, reason = eng.stop, 'A short stop/trail'
                elif held>=MAX_HOLD_TREND: exit_px, reason = closes[i], 'A short time'
            elif eng.pos==3 and can_exit:
                z_hit=(not np.isnan(vwap_z[i])) and (vwap_z[i]>=-VWAP_Z_EXIT)
                if lows[i]<=eng.stop: exit_px, reason = eng.stop, 'B long SL'
                elif (not math.isnan(eng.tp) and highs[i]>=eng.tp) or z_hit: exit_px, reason = (eng.tp if not math.isnan(eng.tp) and highs[i]>=eng.tp else closes[i]), 'B long TP'
                elif held>=MAX_HOLD_RANGE: exit_px, reason = closes[i], 'B long time'
            elif eng.pos==4 and can_exit:
                z_hit=(not np.isnan(vwap_z[i])) and (vwap_z[i]<=VWAP_Z_EXIT)
                if highs[i]>=eng.stop: exit_px, reason = eng.stop, 'B short SL'
                elif (not math.isnan(eng.tp) and lows[i]<=eng.tp) or z_hit: exit_px, reason = (eng.tp if not math.isnan(eng.tp) and lows[i]<=eng.tp else closes[i]), 'B short TP'
                elif held>=MAX_HOLD_RANGE: exit_px, reason = closes[i], 'B short time'
            if exit_px is not None:
                side='long' if eng.pos in (1,3) else 'short'
                px=eng.slip_price(float(exit_px), side, False); fee=eng.fee(eng.qty*px)
                if side=='long':
                    proceeds=eng.qty*px; pnl=proceeds-eng.qty*eng.entry_price-fee-eng.fee(eng.qty*eng.entry_price); eng.cash+=proceeds-fee
                else:
                    pnl=eng.qty*(eng.entry_price-px)-fee-eng.fee(eng.qty*eng.entry_price); eng.cash-=eng.qty*px+fee
                t=eng.trades[-1]; t.exit_time=idx[i]; t.exit_price=px; t.pnl=pnl; t.reason=reason
                eng.equity=eng.cash; eng.pos=0; eng.qty=0.0; eng.last_exit_bar=i; eng.tp=math.nan; eng.trail=math.nan
        can_trade = eng.pos==0 and (i-eng.last_exit_bar>=COOLDOWN_BARS)
        if can_trade:
            side=play=None; stop=tp=trail=math.nan
            if long_a[i]: side,play='long','A'; stop=closes[i]-ATR_SL_TREND*atrs[i]; trail=closes[i]-ATR_TRAIL_TREND*atrs[i]; eng.pos=1
            elif short_a[i]: side,play='short','A'; stop=closes[i]+ATR_SL_TREND*atrs[i]; trail=closes[i]+ATR_TRAIL_TREND*atrs[i]; eng.pos=2
            elif long_b[i]: side,play='long','B'; stop=closes[i]-ATR_SL_RANGE*atrs[i]; tp=closes[i]+ATR_TP_RANGE*atrs[i]; eng.pos=3
            elif short_b[i]: side,play='short','B'; stop=closes[i]+ATR_SL_RANGE*atrs[i]; tp=closes[i]-ATR_TP_RANGE*atrs[i]; eng.pos=4
            if side is not None:
                entry=eng.slip_price(float(closes[i]), side, True); qty=eng.size_from_stop(entry, float(stop))
                if qty<=0: eng.pos=0
                else:
                    fee=eng.fee(qty*entry)
                    if side=='long':
                        cost=qty*entry+fee
                        if cost>eng.cash: qty=max(0.0,(eng.cash-fee)/entry); fee=eng.fee(qty*entry); cost=qty*entry+fee
                        eng.cash-=cost
                    else: eng.cash+=qty*entry-fee
                    eng.entry_bar=i; eng.entry_price=entry; eng.stop=float(stop)
                    eng.tp=float(tp) if not (isinstance(tp,float) and math.isnan(tp)) else math.nan
                    eng.trail=float(trail) if not (isinstance(trail,float) and math.isnan(trail)) else math.nan
                    eng.qty=qty; eng.playbook=play; eng.equity=eng.mark_equity(entry)
                    eng.trades.append(Trade(side=side, playbook=play, entry_time=idx[i], entry_price=entry, qty=qty))
        eng.equity=eng.mark_equity(closes[i]); eng.equity_curve.append(eng.equity)
    if eng.pos!=0 and eng.trades:
        px=eng.slip_price(float(closes[-1]), 'long' if eng.pos in (1,3) else 'short', False); fee=eng.fee(eng.qty*px); t=eng.trades[-1]
        if eng.pos in (1,3): eng.cash+=eng.qty*px-fee; t.pnl=eng.qty*(px-eng.entry_price)-fee-eng.fee(eng.qty*eng.entry_price)
        else: eng.cash-=eng.qty*px+fee; t.pnl=eng.qty*(eng.entry_price-px)-fee-eng.fee(eng.qty*eng.entry_price)
        t.exit_time=idx[-1]; t.exit_price=px; t.reason='EOD flatten'; eng.pos=0; eng.equity=eng.cash; eng.equity_curve[-1]=eng.equity
    return eng

eng = run_backtest(feats)
start, end = STARTING_CAPITAL, eng.equity
closed=[t for t in eng.trades if t.exit_time is not None]
wins=[t for t in closed if t.pnl>0]; losses=[t for t in closed if t.pnl<=0]
eq=pd.Series(eng.equity_curve, index=feats.index[:len(eng.equity_curve)])
dd=(eq/eq.cummax()-1).min()*100
bh=(feats['close'].iloc[-1]/feats['close'].iloc[0]-1)*100
print('='*60)
print('STARTING CAPITAL: $' + f'{start:,.2f}')
print('ENDING EQUITY:    $' + f'{end:,.2f}')
print('NET PROFIT:       $' + f'{end-start:,.2f}')
print('RETURN:           ' + f'{(end/start-1)*100:.2f}%')
print('BUY & HOLD:       ' + f'{bh:.2f}%')
print('MAX DRAWDOWN:     ' + f'{dd:.2f}%')
print('TRADES:', len(closed), '| WINS:', len(wins), '| LOSSES:', len(losses))
if closed:
    print('WIN RATE:', f'{100*len(wins)/len(closed):.1f}%')
    print('AVG PnL: $' + f'{np.mean([t.pnl for t in closed]):,.2f}')
print('='*60)

plt.figure(figsize=(12,4)); eq.plot(title='Equity Curve ($100k start)'); plt.ylabel('Equity'); plt.grid(True, alpha=0.3); plt.show()
trades_df=pd.DataFrame([{'side':t.side,'playbook':t.playbook,'entry':t.entry_time,'entry_px':t.entry_price,'exit':t.exit_time,'exit_px':t.exit_price,'pnl':t.pnl,'reason':t.reason} for t in closed])
display(trades_df.tail(20))
trades_df.to_csv('backtest_trades.csv', index=False)
print('Saved backtest_trades.csv')